---
### 🐛 Found a bug or have a suggestion?
[Open a GitHub Issue](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues/new?template=bug_report.md) — describe the notebook name, cell number, and what went wrong.
*Search [existing issues](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues) first to avoid duplicates.*

# 📡 Time Series Forecasting
**fpppy Chapter 8 · Pattern Recognition for the Rest of Us**

> Jump in — each section builds on the last. Run cells top to bottom with Shift+Enter.

### What you'll learn
- Baseline forecasts: naïve, seasonal naïve, mean
- Exponential Smoothing (ETS / Holt-Winters)
- ARIMA — identifying p, d, q
- Forecast evaluation on a held-out test set

### Time: ~55 minutes

In [ ]:
# ⚠️  Run this cell first — sets up data and imports
# Tip: Runtime → Run all  (runs everything top to bottom automatically)

import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': '#f8f9fa',
    'axes.grid': True, 'grid.alpha': 0.4,
    'axes.spines.top': False, 'axes.spines.right': False,
    'font.size': 11
})

from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ── Load / generate data ──────────────────────────────────────────────────────
passengers = pd.read_csv('https://raw.githubusercontent.com/ladataanalytics/pattern-recognition-notebooks/main/data/AirPassengers.csv')
passengers['Month'] = pd.to_datetime(passengers['Month'])
passengers = passengers.set_index('Month')
split_date = '1958-01-01'
train = passengers[passengers.index < split_date]
test  = passengers[passengers.index >= split_date]
print(f'✓ Air Passengers: train={len(train)} test={len(test)}')

print(f'  Python {sys.version.split()[0]} | pandas {pd.__version__}')
print('✓ Setup complete')


## 📐 Part 1 — Baseline Forecasts

In [ ]:
# Implement baseline forecasts
h = len(test)

naive_fc      = pd.Series([train['Passengers'].iloc[-1]] * h, index=test.index)
seasonal_naive = train['Passengers'].iloc[-12:].values  # last 12 months repeated
snaive_fc     = pd.Series(np.tile(seasonal_naive, 2)[:h], index=test.index)
mean_fc       = pd.Series([train['Passengers'].mean()] * h, index=test.index)

def eval_forecast(name, fc, actual):
    mae  = mean_absolute_error(actual, fc)
    rmse = np.sqrt(mean_squared_error(actual, fc))
    mape = np.mean(np.abs((actual - fc) / actual)) * 100
    return {'Model':name, 'MAE':mae, 'RMSE':rmse, 'MAPE%':mape}

results = []
for name, fc in [('Naive', naive_fc), ('Seasonal Naive', snaive_fc), ('Mean', mean_fc)]:
    results.append(eval_forecast(name, fc, test['Passengers']))

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(passengers.index, passengers['Passengers'], color='black', lw=1.5, label='Actual', alpha=0.8)
ax.axvline(test.index[0], color='#888', lw=1.5, ls='--', label='Train/Test split')
for fc, color, name in [(naive_fc,'#e85d20','Naive'),
                         (snaive_fc,'#1e5fa8','Seasonal Naive'),
                         (mean_fc,'#888','Mean')]:
    ax.plot(fc.index, fc, color=color, lw=1.5, ls='--', label=name)
ax.set_title('Baseline Forecasts — Air Passengers')
ax.legend()
plt.tight_layout()
plt.show()

print("\n=== Baseline Performance ===")
print(pd.DataFrame(results).to_string(index=False, float_format='%.2f'))
print("\n📌 Seasonal Naive is surprisingly hard to beat on strongly seasonal data")

## 🔬 Part 2 — Exponential Smoothing (ETS)

In [ ]:
# Fit Holt-Winters (additive and multiplicative)
hw_add  = ExponentialSmoothing(train['Passengers'],
                                trend='add', seasonal='add', seasonal_periods=12).fit()
hw_mult = ExponentialSmoothing(train['Passengers'],
                                trend='add', seasonal='mul', seasonal_periods=12).fit()

fc_add  = hw_add.forecast(h)
fc_mult = hw_mult.forecast(h)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(passengers.index, passengers['Passengers'], color='black', lw=1.5, label='Actual')
ax.axvline(test.index[0], color='#888', lw=1.5, ls='--')
ax.plot(fc_add.index,  fc_add,  color='#1e5fa8', lw=2, label='Holt-Winters Additive')
ax.plot(fc_mult.index, fc_mult, color='#e85d20', lw=2, label='Holt-Winters Multiplicative')
ax.fill_between(test.index,
                fc_mult * 0.92, fc_mult * 1.08,
                alpha=0.15, color='#e85d20', label='~80% interval (multiplicative)')
ax.set_title('Holt-Winters Forecasts — Air Passengers')
ax.legend()
plt.tight_layout()
plt.show()

results += [eval_forecast('HW Additive',       fc_add,  test['Passengers']),
            eval_forecast('HW Multiplicative',  fc_mult, test['Passengers'])]

print("\n=== ETS vs Baseline ===")
print(pd.DataFrame(results).to_string(index=False, float_format='%.2f'))
print("\nSmoothing parameters (multiplicative):")
print(f"  α (level) = {hw_mult.params['smoothing_level']:.3f}")
print(f"  β (trend) = {hw_mult.params['smoothing_trend']:.3f}")
print(f"  γ (seasonal) = {hw_mult.params['smoothing_seasonal']:.3f}")

## 📊 Part 3 — ARIMA Modelling

In [ ]:
# auto_arima — finds optimal (p,d,q)(P,D,Q) automatically
print("Running auto_arima (may take 30-60 seconds)...")
arima_model = auto_arima(
    train['Passengers'],
    seasonal=True,
    m=12,           # monthly seasonality
    stepwise=True,  # faster search
    information_criterion='aic',
    trace=False,
    error_action='ignore',
    suppress_warnings=True
)
print("\nBest ARIMA model found:")
print(arima_model.summary())

## ✅ Part 4 — Forecast Evaluation

In [ ]:
# Forecast with best ARIMA
arima_fc_vals, conf_int = arima_model.predict(n_periods=h, return_conf_int=True)
arima_fc = pd.Series(arima_fc_vals, index=test.index)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(passengers.index, passengers['Passengers'], color='black', lw=1.5, label='Actual')
ax.axvline(test.index[0], color='#888', lw=1.5, ls='--', label='Train/Test split')
ax.plot(arima_fc.index, arima_fc, color='#1a7a45', lw=2, label=f'ARIMA{arima_model.order}{arima_model.seasonal_order}')
ax.fill_between(test.index, conf_int[:,0], conf_int[:,1], alpha=0.15, color='#1a7a45', label='95% CI')
ax.plot(fc_mult.index, fc_mult, color='#e85d20', lw=1.5, ls='--', alpha=0.7, label='HW Multiplicative')
ax.set_title('ARIMA vs Holt-Winters — Air Passengers Forecast')
ax.legend()
plt.tight_layout()
plt.show()

results.append(eval_forecast('ARIMA', arima_fc, test['Passengers']))
print("\n=== Final Model Comparison ===")
df_res = pd.DataFrame(results).sort_values('RMSE')
print(df_res.to_string(index=False, float_format='%.2f'))
print(f"\n🏆 Best model by RMSE: {df_res.iloc[0]['Model']}")

In [ ]:
# Residual diagnostics — good model → white noise residuals
arima_model.plot_diagnostics(figsize=(12, 8))
plt.suptitle('ARIMA Residual Diagnostics\n(residuals should look like white noise)', y=1.01)
plt.tight_layout()
plt.show()
print("\n📌 What to look for:")
print("  Standardized residuals: random scatter around zero, no patterns")
print("  Histogram: approximately normal")
print("  Q-Q plot: points on the diagonal")
print("  Correlogram: no significant spikes (residuals uncorrelated)")

## 💼 Exercise

Using the Air Passengers dataset, fit Holt-Winters with multiplicative seasonality. Generate a 24-month forecast. Compare test RMSE against the naïve seasonal baseline. Does ETS outperform the baseline?

In [ ]:
# ── Exercise workspace ──────────────────────────────────────────────────────
# Write your code here



In [ ]:
# @title 📝 Quiz — ETS & ARIMA Forecasting
# @markdown Answer each question in the box below, then run the AI grading cell.

# @markdown **Q1:** What do the three parameters of ARIMA(p,d,q) each represent?
# @markdown **Q2:** Why can you NOT randomly shuffle train/test split for time series?
# @markdown **Q3:** When would you choose Holt-Winters over ARIMA?
# @markdown **Q4:** What does the 'd' in ARIMA represent and when is d=1 used?
# @markdown **Q5:** If residuals show significant ACF spikes, what does that indicate?

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
answered = sum(1 for v in answers.values() if str(v).strip())
print(f"  {answered}/5 answered — run the AI grading cell below")

In [ ]:
_NB_NAME_="ETS & ARIMA Forecasting"
# @title 📋 Quiz Submission
# @markdown **Click ▶ Run** → copy the output box (click the copy icon in the top-left of the output box, or click ⋮ → Copy) → paste into Gemini in this Colab session.

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"your GitHub username e.g. jsmith42"}

_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

if "answers" not in globals():
    print("Run the quiz cell above first, then re-run this cell.")
else:
    qa = "\n".join(
        f"Q{i}: {str(v).strip() or '(no answer)'}"
        for i, (_, v) in enumerate(answers.items(), 1)
    )
    print(f"Please grade my quiz answers for the \"{_NB_TITLE}\" notebook")
    print(f"from \"Data Pattern Recognition for the Rest of Us\" (based on ISLP).")
    if GITHUB_USERNAME.strip():
        print(f"Student: @{GITHUB_USERNAME.strip()}")
    print()
    print(qa)
    print()
    print("For each question:")
    print("  1. Say CORRECT, PARTIAL, or INCORRECT")
    print("  2. Explain in 2-3 sentences why — what did I get right or wrong")
    print("  3. Give the ideal complete answer so I know exactly what was expected")
    print("  4. If I was wrong or partial, tell me the specific concept to review")
    print("     and where it appears in the notebook (e.g. Part 2, the SHAP beeswarm plot)")
    print()
    print("End with:")
    print("  - Overall grade: Excellent / Good / Needs Review / Incomplete")
    print("  - A short study plan: which questions to re-read and what to focus on")
    print()
    print("After grading, I will paste specific outputs, charts, or code blocks")
    print("from the notebook — please explain them in detail and answer follow-up questions.")


---
### 🐛 Found a bug or have a suggestion?
[Open a GitHub Issue](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues/new?template=bug_report.md) — describe the notebook name, cell number, and what went wrong.
*Search [existing issues](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues) first to avoid duplicates.*

---
*Data Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*